# Part 7 - Compaction and Forgetting: surviving the long haul

*Agents from First Principles, Part 7.*

Part 6 gave the agent typed stores it can write to. Run it long enough and two things break.

- The transcript it re-sends every step keeps growing until it overflows the context window.
- A store that only ever grows fills with stale and duplicate facts until the signal drowns in noise.

An agent that cannot forget cannot run for long. This part adds the two operations a long-lived agent needs: **compress the history** so it still fits, and **forget the memories** that no longer earn their place.

RAG Part 20 left an IOU here. It kept a flat conversational buffer and noted "we will come back to summarizing older turns." This is where we come back to it. But be precise about what is different, because it is easy to conflate two operations:

- **P20 CONDENSATION** rewrites the latest QUESTION into a standalone one so retrieval works ("what about its battery?" -> "what is the battery life of the earbuds?").
- **COMPACTION** compresses the HISTORY so the run fits the window. Different input, different output, different purpose.

Three mechanisms, one per axis of growth:

1. **COMPACTION** (the window axis), the Anthropic-style hot/warm/cold idea by hand. When the token budget is crossed, keep the last N turns HOT (verbatim), fold the middle into a WARM rolling summary that keeps decisions and tool outputs and drops chatter, and fold the oldest WARM into a single COLD broad summary. The token bar drops back under budget and the run continues.
2. **FORGETTING** (the store axis). A memory is not equally worth keeping forever. READ time: rank by importance x recency x access, surface the top few. WRITE time: importance DECAYS with age, a SUPERSEDING fact retires the one it replaces, and when the store is over capacity the lowest-scoring memory is EVICTED.
3. **CONSOLIDATION** (a sleep-time pass). Periodically distill the raw EPISODIC log into durable SEMANTIC facts and PROCEDURAL rules, then prune the consumed episodes. This is the long-horizon version of Part 6's stores: events become knowledge.

**Continuity**: same world (Dana, `ORD-3300`, the `$200` order, the 10% restocking fee -> `$180`, the Globex earbuds 2-year warranty). Tokens are estimated deterministically by word count and time is a logical clock, so the demo is reproducible. `generate()` is the real-LLM summarizer one flag away.

> **Runs with no network, no API key, and no third-party dependency.** The deterministic rule compactor/consolidator is the offline default; `generate()` is the real-LLM path one env flag away. Set `OPENAI_API_KEY` to see the real banner; the code always falls through to the deterministic rules so output stays reproducible.

Companion script: `compaction_and_forgetting.py`

## Setup

A single standard-library import does the work: `os` lets us check for an API key without ever requiring one. Everything else is plain Python, so every cell runs fully offline, exactly as in Parts 1-6. The token "estimate" is just a word count: a real system uses the model's tokenizer, but word count is reproducible and close enough to show the mechanism, which is the whole reason the output is deterministic.

In [ ]:
import os


def est_tokens(text):
    """A deterministic token estimate: word count. Real systems use the model's
    tokenizer; word count is reproducible and close enough to show the mechanism."""
    return len(text.split())


print("ready -- no network, no API key, no third-party dependency required")

## The distinction this whole part hangs on: condensation is NOT compaction

Before any code, fix the one confusion that sinks people here. RAG Part 20 and this part both touch "older turns," but they do opposite jobs:

| | P20 CONDENSATION | P7 COMPACTION |
|---|---|---|
| **Input** | the latest user question + a little history | the entire growing history |
| **Output** | one rewritten, standalone question | a compressed transcript (hot/warm/cold) |
| **Purpose** | make retrieval work on a follow-up | keep the run inside the context window |
| **Example** | "what about its battery?" -> "what is the battery life of the earbuds?" | 10 raw turns -> "[2 earlier turns summarized]" + a warm line + the last 2 turns |

Condensation rewrites the QUESTION so a single retrieval call lands. Compaction rewrites the HISTORY so the next model call still fits. They are not interchangeable, and neither is the long-context tradeoff RAG Parts 11 and 16 weighed (whether to retrieve at all versus stuff a huge window). Keep the three apart and the rest of this part reads cleanly.

## Step 1 - The world to compact: a support session, turn by turn

A long support session is just a list of `(role, text, salient)` turns. The `salient` flag is the load-bearing piece: a tool output or a decision is salient and must survive a compaction; pure chatter ("thanks for checking that," "seems fair enough") is not and can be dropped. This is the same Dana/`ORD-3300` refund conversation the series has carried since Part 5, now run long enough that it will overflow.

`BUDGET` is small (40 "tokens") on purpose, so the window crosses it partway through and we can watch compaction fire repeatedly. `HOT_N` is how many of the most recent turns we keep verbatim no matter what.

In [ ]:
BUDGET = 40          # token budget for the live window (small, to trigger compaction)
HOT_N = 2            # keep the last N turns verbatim

SESSION = [
    ("user", "Hi, I'm Dana, I'd like to ask about order ORD-3300.", False),
    ("tool", "search_policy -> refunds within 30 days of purchase.", True),
    ("user", "Oh interesting, thanks for checking that for me.", False),
    ("user", "It has actually been about 40 days since I bought it.", False),
    ("tool", "search_policy -> after the window, a 10% restocking fee applies.", True),
    ("user", "Got it, that seems fair enough I suppose.", False),
    ("user", "So what would my refund be on the $200 order?", False),
    ("tool", "calculator -> $180.00 (200 * 0.9 after the fee).", True),
    ("user", "Perfect. Could you also check the earbuds warranty?", False),
    ("tool", "search_products -> Globex earbuds carry a 2-year limited warranty.", True),
]


def render(role, text):
    return f"{role}: {text}"


print(f"SESSION: {len(SESSION)} turns. BUDGET={BUDGET} tokens, HOT_N={HOT_N}.")

## Step 2 - The three tiers and how the window is measured

Compaction keeps three tiers, hottest to coldest:

- **HOT**: the last `HOT_N` turns, kept verbatim. This is what the model needs in full.
- **WARM**: a rolling summary of older SALIENT turns (the decisions and tool outputs), one line each.
- **COLD**: a single lossy line saying how many ancient turns were summarized away. The far past costs almost nothing to carry.

`cold_gist(n)` renders the cold tier; a real system would keep a short paraphrase here, but the point is the same: the oldest history collapses to near-zero cost. `window_tokens` measures the live window the model would actually see: cold gist + every warm line + every hot turn rendered. That number is what we compare against `BUDGET`.

In [ ]:
def cold_gist(n):
    """The COLD tier is a single lossy line: how many older turns were summarized
    away. A real system would keep a short paraphrase here; the point is that the
    far past costs almost nothing to carry."""
    return f"[{n} earlier turns summarized]" if n else ""


def window_tokens(hot, warm, cold_n):
    t = est_tokens(cold_gist(cold_n))
    t += sum(est_tokens(w) for w in warm)
    t += sum(est_tokens(render(r, x)) for r, x, _ in hot)
    return t


print("cold_gist, window_tokens ready.")

## Step 3 - `compact()`: keep HOT verbatim, fold the middle WARM, drop chatter

This is the core operation. When the window crosses budget, `compact()` runs:

1. Keep the last `HOT_N` turns verbatim (`keep`).
2. Walk the older turns: a SALIENT one (a decision or tool output) is appended to the WARM summary; a chatter turn is DROPPED and counted.
3. If the window is still over budget, pop the OLDEST warm line into the cold gist (incrementing `cold_n`) until it fits.

It guarantees `window <= BUDGET` as long as the hot turns alone fit, and returns the new `(hot, warm, cold_n, dropped)`. Note what survives versus what vanishes: the `$180` calculation and the warranty lookup persist; "thanks for checking" and "seems fair enough" do not. Compaction is lossy on purpose, but it loses the chatter first.

In [ ]:
def compact(hot, warm, cold_n):
    """Keep the last HOT_N turns verbatim; fold older SALIENT turns into the warm
    summary and drop chatter; then fold the OLDEST warm items into the cold gist
    until the window is back under budget. Guarantees window <= BUDGET (as long as
    the hot turns alone fit). Returns (hot, warm, cold_n, dropped)."""
    keep = hot[-HOT_N:]
    dropped = 0
    for role, text, salient in hot[:-HOT_N]:
        if salient:
            warm.append(text)                          # a decision/tool output: keep it
        else:
            dropped += 1                               # chatter: drop it
    while window_tokens(keep, warm, cold_n) > BUDGET and warm:
        warm.pop(0)                                    # oldest warm decision -> cold gist
        cold_n += 1
    return keep, warm, cold_n, dropped


print("compact ready.")

## Step 4 - `run_compaction()`: replay the session and compact on overflow

The driver replays the session one turn at a time. After each turn it measures the window; if it is over budget it prints `OVER` and calls `compact()`, then reports how many chatter turns were dropped and the new window size. This is exactly the loop a long-running agent would run before every model call: append the new turn, check the budget, compact if needed, then send. At the end it prints the final three-tier window, which is all the model still sees of a ten-turn conversation.

In [ ]:
def run_compaction():
    hot, warm, cold_n = [], [], 0
    for i, (role, text, salient) in enumerate(SESSION, start=1):
        hot.append((role, text, salient))
        tokens = window_tokens(hot, warm, cold_n)
        flag = "OVER" if tokens > BUDGET else "ok"
        print(f"  turn {i:>2}: window {tokens:>2}/{BUDGET} {flag}  (+ {render(role, text)[:44]})")
        if tokens > BUDGET:
            hot, warm, cold_n, dropped = compact(hot, warm, cold_n)
            after = window_tokens(hot, warm, cold_n)
            print(f"          -> COMPACT: dropped {dropped} chatter turn(s), folded older "
                  f"decisions warm/cold; window now {after}/{BUDGET}")
    print("\n  Final window state (this is all the model still sees):")
    if cold_n:
        print(f"    COLD : {cold_gist(cold_n)}")
    print(f"    WARM : {warm}")
    print(f"    HOT  : {[render(r, x) for r, x, _ in hot]}")


print("run_compaction ready.")

## Step 5 - Forgetting, part one: the `Fact` and its score

Compaction handled the window axis. The store axis is FORGETTING. Part 6 gave the agent a SEMANTIC store, but it only ever grew. Here each fact carries the metadata a forgetting policy needs: an `importance`, the `last_touch` time, an `access` count, and a `superseded` flag.

Two functions turn that metadata into a single number:

- `recency(fact, now)` is exponential decay with a half-life: a fact loses half its weight every `HALF_LIFE` logical time units since it was last touched.
- `score(fact, now)` is `importance x recency x access-boost`. A fact that is intrinsically important, recently touched, and frequently read scores high; a stale, untouched, trivial fact scores near zero.

This is the read-time ranking, and (with a fresh `now`) it is also the eviction criterion. One score, two uses.

In [ ]:
CAPACITY = 4         # the store holds at most this many active facts
HALF_LIFE = 4.0      # logical time units for recency to halve


class Fact:
    def __init__(self, key, text, importance, created):
        self.key = key
        self.text = text
        self.importance = importance
        self.last_touch = created
        self.access = 0
        self.superseded = False


def recency(fact, now):
    return 0.5 ** ((now - fact.last_touch) / HALF_LIFE)


def score(fact, now):
    return fact.importance * recency(fact, now) * (1 + 0.1 * fact.access)


print("Fact, recency, score ready.")

## Step 6 - Forgetting, part two: `read_top` (rank), `add_fact` (decay / supersede / evict)

`read_top` is the READ path: drop superseded facts, sort the rest by `score`, return the top `k`. The model never sees the whole store, only the few facts that currently earn their place.

`add_fact` is the WRITE path, and it carries three of the four forgetting behaviors at once:

- **Supersession**: if `supersedes` names an existing key, that older fact is flagged `superseded` (retired, not deleted). A new "prefers phone" retires the old "prefers email."
- **Eviction**: after appending, if the active set exceeds `CAPACITY`, the lowest-scoring active fact is evicted (also via `superseded`) and returned, so the caller can report it.
- **Decay** is implicit: it happens through `score` as `now` advances; nobody has to actively age a fact.

In [ ]:
def read_top(store, now, k=3):
    active = [f for f in store if not f.superseded]
    ranked = sorted(active, key=lambda f: -score(f, now))
    return ranked[:k]


def add_fact(store, key, text, importance, now, supersedes=None):
    if supersedes:
        for f in store:
            if f.key == supersedes and not f.superseded:
                f.superseded = True
    store.append(Fact(key, text, importance, now))
    # Evict the lowest-scoring active fact if over capacity.
    active = [f for f in store if not f.superseded]
    if len(active) > CAPACITY:
        victim = min(active, key=lambda f: score(f, now))
        victim.superseded = True
        return victim
    return None


print("read_top, add_fact ready.")

## Step 7 - `run_forgetting()`: rank, let time pass, supersede, evict

The driver seeds four facts about Dana, then walks the lifecycle:

1. **Read at t=0**: rank by raw importance (everything was just touched, so recency is 1).
2. **Read at t=6**: recency has decayed every score; the `name` fact got read twice, so its access boost partly offsets the decay. The ordering still holds but the absolute scores have shrunk.
3. **Supersession**: "prefers phone" arrives and retires "prefers email." Both stay in the store; one is tagged retired.
4. **Eviction**: a fifth, low-value fact ("clicked a promo email once") pushes the active set over `CAPACITY=4`, so the weakest active fact, the stale "raining today," is evicted.

Watch the chatter fact ("raining today") lose first, exactly as chatter lost first in compaction. The two axes use different machinery but the same instinct: keep what earns its place.

In [ ]:
def run_forgetting():
    now = 0
    store = []
    add_fact(store, "name", "the user is Dana", 0.9, now)
    add_fact(store, "contact", "Dana prefers email contact", 0.6, now)
    add_fact(store, "order", "Dana's order is ORD-3300 ($200)", 0.7, now)
    add_fact(store, "chitchat", "Dana mentioned it is raining today", 0.2, now)

    print("  Read-time ranking at t=0 (importance x recency x access):")
    for f in read_top(store, now, k=4):
        print(f"    {score(f, now):.2f}  {f.text}")

    now = 6                                            # time passes; recency decays
    store[0].access += 2                               # 'name' got read a couple of times
    print(f"\n  At t={now}, importance has decayed with age; 'name' was accessed twice:")
    for f in read_top(store, now, k=4):
        print(f"    {score(f, now):.2f}  {f.text}")

    print("\n  SUPERSESSION: Dana says to use the phone instead of email.")
    add_fact(store, "contact", "Dana prefers phone contact", 0.6, now, supersedes="contact")
    for f in store:
        if f.key == "contact":
            tag = "retired" if f.superseded else "active"
            print(f"    [{tag}] {f.text}")

    print("\n  EVICTION: a 5th fact arrives but capacity is 4; the weakest is evicted.")
    victim = add_fact(store, "promo", "Dana clicked a promo email once", 0.15, now)
    print(f"    evicted (lowest score): {victim.text}")
    print("  Active store now:")
    for f in read_top(store, now, k=9):
        print(f"    {score(f, now):.2f}  {f.text}")


print("run_forgetting ready.")

## Step 8 - Consolidation: a sleep-time pass turns raw episodes into knowledge

Compaction shrinks the window; forgetting prunes the store. Consolidation is the third move, and it runs offline, between sessions, like sleep. It reads the raw EPISODIC log (what HAPPENED, the event stream from Part 6) and distills it into the durable stores: SEMANTIC facts (what is TRUE) and PROCEDURAL rules (how to DO a task). Then it prunes the consumed episodes, because once "the user confirmed their name is Dana" has become `user.name = Dana`, the raw event no longer earns its place.

`consolidate` here is a transparent rule pass keyed on the same cheap signals an LLM would weigh: a "name is" event becomes a semantic fact; a "restocking fee" event becomes a procedural rule. It de-duplicates while preserving order. A real system swaps this body for one `generate()` call; the contract is identical either way.

The driver `run_consolidation()` then prints the raw log, runs `consolidate`, and reports the distilled SEMANTIC facts, PROCEDURAL rules, and the count of raw events cleared. Four messy episodes collapse into one durable fact and one reusable rule, which is the long-horizon version of Part 6's typed stores: events become knowledge, and the knowledge is small enough to keep forever.

In [ ]:
EPISODIC_LOG = [
    "t1: user asked whether a return after the window is allowed",
    "t2: tool said after the window a 10% restocking fee applies",
    "t3: user accepted the $180.00 refund on the $200 order",
    "t4: user confirmed their name is Dana",
]


def consolidate(episodic):
    semantic, procedural = [], []
    for event in episodic:
        low = event.lower()
        if "name is" in low:
            semantic.append("user.name = Dana")
        if "restocking fee" in low:
            procedural.append("returns after the 30-day window: apply a 10% restocking fee")
    # de-duplicate while preserving order
    semantic = list(dict.fromkeys(semantic))
    procedural = list(dict.fromkeys(procedural))
    return semantic, procedural


def run_consolidation():
    print("  Raw episodic log (what happened):")
    for e in EPISODIC_LOG:
        print(f"    {e}")
    semantic, procedural = consolidate(EPISODIC_LOG)
    print("\n  After the sleep-time consolidation pass:")
    print(f"    -> SEMANTIC facts   : {semantic}")
    print(f"    -> PROCEDURAL rules : {procedural}")
    print(f"    -> episodic log pruned: {len(EPISODIC_LOG)} raw events distilled and cleared")


print(f"EPISODIC_LOG: {len(EPISODIC_LOG)} raw events. consolidate, run_consolidation ready.")

## Step 10 - `generate()`: the real LLM summarizer (reference shape only)

Same device as Parts 1-6. In production the summarizer and the consolidator are an LLM: you hand it the history to compress or the episodes to distill, and it returns the summary. `generate()` is the single swappable call that lights up the real path; the offline demo never calls it, because the deterministic rule compactor/consolidator is the source of truth for everything in the output. SDK names and model IDs move fast, so only `generate()` would need edits to go live.

In [ ]:
def generate(prompt):
    """REAL path: ask a hosted LLM to summarize history or consolidate. Unused offline."""
    from openai import OpenAI
    client = OpenAI()                               # reads OPENAI_API_KEY
    resp = client.chat.completions.create(
        model="gpt-4o-mini",                        # a small chat model; check names
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content

# Anthropic / Claude alternative. Swap in for generate() above:
#
# def generate(prompt):
#     from anthropic import Anthropic
#     client = Anthropic()                            # reads ANTHROPIC_API_KEY
#     resp = client.messages.create(
#         model="claude-opus-4-8",                    # check current model names
#         max_tokens=1024,
#         messages=[{"role": "user", "content": prompt}],
#     )
#     return resp.content[0].text


if os.environ.get("OPENAI_API_KEY"):
    print("[summarizer] OPENAI_API_KEY set; the real LLM would summarize via generate(). "
          "Falling through to the deterministic rules so output is reproducible.")
else:
    print("[summarizer] no OPENAI_API_KEY; using deterministic rule compactor (offline default)")

## Demo - run compaction, then forgetting, then consolidation

Everything below runs fully offline. The demo has three sections, one per mechanism. First, **compaction** replays the ten-turn session and folds it down to a three-tier window every time the budget is crossed. Second, **forgetting** ranks facts by score, lets time decay them, supersedes a stale preference, and evicts the weakest when the store overflows. Third, **consolidation** distills the raw episodic log into durable facts and rules. The banner restates the P20-condensation-versus-compaction distinction so it stays front of mind.

In [ ]:
bar = "=" * 72
print(bar)
print("SURVIVING THE LONG HAUL  -  compaction and forgetting")
print(bar)
if os.environ.get("OPENAI_API_KEY"):
    print("[summarizer] OPENAI_API_KEY set; the real LLM would summarize via generate(). "
          "Falling through to the deterministic rules so output is reproducible.")
else:
    print("[summarizer] no OPENAI_API_KEY; using deterministic rule compactor (offline default)")

print("\nNOT the same as RAG P20 condensation: P20 rewrites the QUESTION into a standalone")
print("one; COMPACTION compresses the HISTORY to fit the window. Different operations.")

print("\n" + "-" * 72)
print("COMPACTION: keep the last turns HOT, fold the middle into a WARM summary.")
print("-" * 72)
run_compaction()

print("\n" + bar)
print("FORGETTING: score by importance x recency x access; decay, supersede, evict.")
print(bar)
run_forgetting()

print("\n" + bar)
print("CONSOLIDATION: a sleep-time pass turns raw episodes into durable knowledge.")
print(bar)
run_consolidation()

print("\n" + bar)
print("Done. An agent that runs for a long time must do two things a one-shot pipeline")
print("never had to: COMPRESS the history so it still fits the window (hot/warm/cold,")
print("distinct from P20's question condensation), and FORGET the memories that no")
print("longer earn their place (decay, supersession, eviction), distilling raw episodes")
print("into durable facts and rules. Growth without forgetting is just slower failure.")
print(bar)

## Wrap-up

Part 6's typed stores were correct but unbounded: every turn lengthened the transcript and every write grew the store, and context is finite. This part added the other half of a memory system, the half that decides what to keep.

- **Compaction** (the window axis) keeps the last turns HOT verbatim, folds older salient turns into a WARM rolling summary, and collapses the oldest into a single COLD line, dropping chatter along the way. A ten-turn session that overflowed a 40-token budget five times ended as a three-line window the model can still reason over. This is NOT RAG Part 20's condensation: that rewrites the QUESTION for retrieval; this compresses the HISTORY for the window.
- **Forgetting** (the store axis) scores each fact by importance x recency x access, then DECAYS importance with age, SUPERSEDES a stale fact with its replacement, and EVICTS the weakest when the store is over capacity. The chatter ("raining today") lost first, just as it did in compaction.
- **Consolidation** (a sleep-time pass) distilled four raw episodes into one durable SEMANTIC fact and one PROCEDURAL rule, then pruned the consumed log. Events became knowledge, the long-horizon version of Part 6's stores.

Growth without forgetting is just slower failure. An agent that compresses its history and prunes its memory can run for a long time; one that only appends will eventually drown in its own context.

**Part 8 - Budgets and the circuit breaker** is next. A long-running agent that survives the window and the store still has a third way to fail: it can loop, oscillate, or burn unbounded steps and tokens chasing a goal it will never reach. Part 8 adds step and token BUDGETS, LOOP DETECTION, and a CIRCUIT BREAKER, so the agent stops itself before it runs forever.